# Hoja de Trabajo 2
Juan Diego Solís Martínez - 23720

Victor Manuel Pérez Chávez - 23714


## Task 1: Implementación completa
### Instalación y dependencias

In [1]:
!pip install torch torchvision matplotlib requests pillow --quiet

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
import os
from PIL import Image
from io import BytesIO
import time

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

# Dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

Usando dispositivo: cpu


### Parámetros fijos

In [3]:
# Task 1.1 - Parámetros fijos según la hoja de trabajo
Z_DIM = 100
IMG_SIZE = 64
IMG_CHANNELS = 3
FEATURES_G = 64
FEATURES_D = 64

# Task 1.2 - Parámetros de entrenamiento (DCGAN)
BATCH_SIZE = 32
NUM_EPOCHS = 50
LR = 2e-4
BETAS = (0.5, 0.999)

print('Hiperparámetros configurados correctamente.')

Hiperparámetros configurados correctamente.


### Dataset: Descarga de sprites desde el PokeAPI

In [4]:
def download_pokemon_sprites(save_dir='pokemon_sprites', max_pokemon=898):
    os.makedirs(save_dir, exist_ok=True)

    base_url = "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/"
    downloaded = 0
    failed = 0

    print(f'Descargando sprites de {max_pokemon} Pokémon:')

    for pokemon_id in range(1, max_pokemon + 1):
        filepath = os.path.join(save_dir, f'{pokemon_id}.png')

        if os.path.exists(filepath):
            downloaded += 1
            continue

        url = f"{base_url}{pokemon_id}.png"

        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content)).convert('RGBA')

                background = Image.new('RGB', img.size, (255, 255, 255))
                background.paste(img, mask=img.split()[3])
                background.save(filepath)
                downloaded += 1

            else:
                failed += 1

        except Exception as e:
            failed += 1

        if pokemon_id % 100 == 0:
            print(f' - Progreso: {pokemon_id}/{max_pokemon} — descargados: {downloaded}, fallidos: {failed}')
        time.sleep(0.05)

    print(f'\nDescarga completada: {downloaded} sprites, {failed} fallidos.')
    return save_dir

sprite_dir = download_pokemon_sprites() # Descargar sprites

Descargando sprites de 898 Pokémon:
 - Progreso: 100/898 — descargados: 100, fallidos: 0
 - Progreso: 200/898 — descargados: 200, fallidos: 0
 - Progreso: 300/898 — descargados: 300, fallidos: 0
 - Progreso: 400/898 — descargados: 400, fallidos: 0
 - Progreso: 500/898 — descargados: 500, fallidos: 0
 - Progreso: 600/898 — descargados: 600, fallidos: 0
 - Progreso: 700/898 — descargados: 700, fallidos: 0
 - Progreso: 800/898 — descargados: 800, fallidos: 0

Descarga completada: 898 sprites, 0 fallidos.


In [5]:
class PokemonDataset(Dataset):
    # Dataset de sprites de Pokémon.
    # Aplica transformaciones: r000esize a 64x64, normalización a [-1, 1]
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_files = [
            f for f in os.listdir(image_dir)
            if f.endswith('.png') or f.endswith('.jpg')
        ]
        print(f'Dataset: {len(self.image_files)} imágenes encontradas.')

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

# Transformaciones: resize + tensor + normalización [-1, 1]
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

dataset = PokemonDataset(sprite_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

print(f'DataLoader: {len(dataloader)} batches de tamaño {BATCH_SIZE}')

Dataset: 898 imágenes encontradas.
DataLoader: 28 batches de tamaño 32


## Task 1.1 | Arquitecturas del Generador y Discriminador

In [6]:
# Inicialización de pesos
# - Capas Conv/ConvTranspose: N(0, 0.02)
# - BatchNorm: gamma ~ N(1, 0.02), beta = 0
def initialize_weights(model):
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight.data, 0.0, 0.02)

        elif isinstance(m, nn.BatchNorm2d):
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0)

In [7]:
# Generador DCGAN
# Entrada: z ∈ R^100 con forma (batch, 100, 1, 1)
# Salida: imagen con forma (batch, 3, 64, 64)
class Generator(nn.Module):
    def __init__(self, z_dim, img_channels, features_g):
        super(Generator, self).__init__()

        self.net = nn.Sequential(
            # Capa 1: z (B,100,1,1) → (B, FG*8, 4, 4)
            # Primera capa: sin padding, stride=1 para expandir desde 1x1
            self._block(z_dim, features_g * 8, kernel_size=4, stride=1, padding=0),

            # Capa 2: (B, FG*8, 4, 4) → (B, FG*4, 8, 8)
            self._block(features_g * 8, features_g * 4, kernel_size=4, stride=2, padding=1),

            # Capa 3: (B, FG*4, 8, 8) → (B, FG*2, 16, 16)
            self._block(features_g * 4, features_g * 2, kernel_size=4, stride=2, padding=1),

            # Capa 4: (B, FG*2, 16, 16) → (B, FG, 32, 32)
            self._block(features_g * 2, features_g, kernel_size=4, stride=2, padding=1),

            # Capa 5 (salida): (B, FG, 32, 32) → (B, 3, 64, 64)
            nn.ConvTranspose2d(features_g, img_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    # Bloque: ConvTranspose2d -> BatchNorm2d -> ReLU
    def _block(self, in_channels, out_channels, kernel_size, stride, padding):
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, z):
        return self.net(z)

In [8]:
# Discriminador DCGAN
# Entrada: imagen con forma (batch, 3, 64, 64)
# Salida:  escalar por imagen con forma (batch,)
class Discriminator(nn.Module):
    def __init__(self, img_channels, features_d):
        super(Discriminator, self).__init__()

        self.net = nn.Sequential(
            # Capa 1: (B, 3, 64, 64) -> (B, FD, 32, 32) | SIN BatchNorm
            nn.Conv2d(img_channels, features_d, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # Capa 2: (B, FD, 32, 32) -> (B, FD*2, 16, 16)
            self._block(features_d, features_d * 2, kernel_size=4, stride=2, padding=1),

            # Capa 3: (B, FD*2, 16, 16) -> (B, FD*4, 8, 8)
            self._block(features_d * 2, features_d * 4, kernel_size=4, stride=2, padding=1),

            # Capa 4: (B, FD*4, 8, 8) → (B, FD*8, 4, 4)
            self._block(features_d * 4, features_d * 8, kernel_size=4, stride=2, padding=1),

            # Capa 5 (salida): (B, FD*8, 4, 4) -> (B, 1, 1, 1)
            # Sin BatchNorm, Sigmoid para clasificación binaria
            nn.Conv2d(features_d * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )

    # Bloque Conv2d -> BatchNorm2d -> LeakyReLU(0.2)
    def _block(self, in_channels, out_channels, kernel_size, stride, padding):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True)
        )

    def forward(self, x):
        return self.net(x).view(-1)

In [9]:
# Instanciar modelos
G = Generator(Z_DIM, IMG_CHANNELS, FEATURES_G).to(device)
D = Discriminator(IMG_CHANNELS, FEATURES_D).to(device)

# Aplicar inicialización de pesos
initialize_weights(G)
initialize_weights(D)

# Verificar formas
z_test = torch.randn(4, Z_DIM, 1, 1).to(device)

assert G(z_test).shape == (4, 3, 64, 64), "Forma del generador incorrecta"
assert D(torch.randn(4, 3, 64, 64).to(device)).shape == (4,), "Forma del discriminador incorrecta"

print('Verificación de formas OK')
print(f' - G(z).shape = {G(z_test).shape}   (esperado: (4, 3, 64, 64))')
print(f' - D(x).shape = {D(torch.randn(4,3,64,64).to(device)).shape}   (esperado: (4,))')

Verificación de formas OK
 - G(z).shape = torch.Size([4, 3, 64, 64])   (esperado: (4, 3, 64, 64))
 - D(x).shape = torch.Size([4])   (esperado: (4,))
